In [0]:
display(
    dbutils.fs.ls("/Volumes/workspace/default/dsp_pipeline")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dsp_pipeline/advertisers.csv,advertisers.csv,401,1784447296000
dbfs:/Volumes/workspace/default/dsp_pipeline/bronze/,bronze/,0,1784562017050
dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,events_day1.csv,3094332,1784447300000
dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,events_day2.csv,4048441,1784447301000


## Bronze Layer – Raw Data Ingestion

Ingest raw vendor files as-is, with no cleaning or deduplication. 

Only metadata columns (`pipeline_ingest_time`, `source_file`) are added for lineage. 

v1/v2 schema drift is preserved by keeping the two event files as separate bronze tables — reconciliation happens explicitly in Silver, not silently here.

In [0]:
raw_path = "/Volumes/workspace/default/dsp_pipeline"
bronze_path = "/Volumes/workspace/default/dsp_pipeline/bronze"

print(raw_path)
print(bronze_path)

/Volumes/workspace/default/dsp_pipeline
/Volumes/workspace/default/dsp_pipeline/bronze


In [0]:
from pyspark.sql.functions import col, current_timestamp

def read_raw(filename):
    """Read a vendor CSV with header/schema inference, keeping source file path for lineage."""
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{raw_path}/{filename}")
        .select("*", "_metadata")
    )

events_day1_raw = read_raw("events_day1.csv")
events_day2_raw = read_raw("events_day2.csv")
advertisers_raw = read_raw("advertisers.csv")

### Schema drift: v1 (day1) vs v2 (day2)

- `spend` (v1) is renamed to `media_cost` (v2) — same meaning, different column name.

- `viewability` is new in v2, not present in v1.

- All other columns (`event_id`, `event_type`, `event_time`, `ingest_time`, 
  `advertiser_id`, `campaign_id`, `placement`, `device`, `geo`, `currency`) are unchanged.

Bronze keeps day1 and day2 as separate tables to preserve raw fidelity exactly as the 
vendor sent it. 

The rename/reconciliation (`media_cost` → `spend`, filling `viewability` 
as null for v1 rows) happens in Silver, where the unified schema is defined.

In [0]:
print("=== v1 (day1) schema ===")
events_day1_raw.printSchema()

print("=== v2 (day2) schema ===")
events_day2_raw.printSchema()

print("=== advertisers schema ===")
advertisers_raw.printSchema()

=== v1 (day1) schema ===
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingest_time: timestamp (nullable = true)
 |-- advertiser_id: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- placement: string (nullable = true)
 |-- device: string (nullable = true)
 |-- geo: string (nullable = true)
 |-- spend: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- _metadata: struct (nullable = false)
 |    |-- file_path: string (nullable = false)
 |    |-- file_name: string (nullable = false)
 |    |-- file_size: long (nullable = false)
 |    |-- file_block_start: long (nullable = false)
 |    |-- file_block_length: long (nullable = false)
 |    |-- file_modification_time: timestamp (nullable = false)

=== v2 (day2) schema ===
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingest_time:

In [0]:
def add_bronze_metadata(df):
    """Attach ingest timestamp and source file path for auditability — no other changes."""
    return (
        df
        .withColumn("pipeline_ingest_time", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .drop("_metadata")
    )

bronze_events_day1 = add_bronze_metadata(events_day1_raw)
bronze_events_day2 = add_bronze_metadata(events_day2_raw)
bronze_advertisers = add_bronze_metadata(advertisers_raw)

In [0]:
bronze_events_day1.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_events_day1")
bronze_events_day2.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_events_day2")
bronze_advertisers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_advertisers")

In [0]:
# Sanity check only — row counts + samples, not part of the pipeline logic.
for t in ["bronze_events_day1", "bronze_events_day2", "bronze_advertisers"]:
    n = spark.table(f"workspace.default.{t}").count()
    print(f"{t}: {n} rows")

display(spark.table("workspace.default.bronze_events_day1").limit(5))
display(spark.table("workspace.default.bronze_events_day2").limit(5))

bronze_events_day1: 26118 rows
bronze_events_day2: 31498 rows
bronze_advertisers: 10 rows


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,spend,currency,pipeline_ingest_time,source_file
evt_7e681157,impression,2026-06-01T03:11:56Z,2026-06-01T09:17:54.000Z,adv_1151,cmp_1151_1,banner_300x250,mobile,IN,0.0194,USD,2026-07-20T15:40:45.242Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7d680fc4,impression,2026-06-01T06:32:35Z,2026-06-01T05:33:13.000Z,adv_1186,cmp_1186_2,interstitial,ctv,IN,0.0102,USD,2026-07-20T15:40:45.242Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_8068147d,click,2026-06-01T18:56:56Z,2026-06-01T01:16:09.000Z,adv_1151,cmp_1151_3,native_feed,mobile,GB,0.0212,INR,2026-07-20T15:40:45.242Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7f6812ea,impression,2026-06-01T20:16:21Z,2026-06-01T03:29:28.000Z,adv_1172,cmp_1172_2,preroll,ctv,DE,0.018,USD,2026-07-20T15:40:45.242Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7a680b0b,impression,2026-06-01T07:43:12Z,2026-06-01T08:48:22.000Z,adv_1165,cmp_1165_1,banner_728x90,desktop,GB,0.0189,USD,2026-07-20T15:40:45.242Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,media_cost,currency,viewability,pipeline_ingest_time,source_file
evt_a815d507,impression,2026-06-04 18:50:42+05:30,2026-06-04T00:18:44.000Z,adv_1144,cmp_1144_2,interstitial,desktop,US,0.0185,USD,0.68,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a815d507,impression,2026-06-04 18:50:42+05:30,2026-06-04T00:18:44.000Z,adv_1144,cmp_1144_2,interstitial,desktop,US,0.0185,USD,0.68,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a715d374,impression,2026-06-04 20:03:11+05:30,2026-06-04T06:56:34.000Z,adv_1151,cmp_1151_1,interstitial,ctv,DE,0.0138,USD,0.48,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_aa15d82d,impression,2026-06-04 04:57:18+05:30,2026-06-04T19:06:57.000Z,adv_1172,cmp_1172_3,preroll,mobile,unknown,0.0131,USD,0.54,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a915d69a,impression,2026-06-04 10:59:30+05:30,2026-06-04T09:08:41.000Z,adv_1137,cmp_1137_2,banner_728x90,mobile,SG,0.0013,USD,0.87,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv


## Silver Layer – Unified, Clean Events

Reconcile v1/v2 schema drift into one events table.

Deduplicate (exact duplicates and 
re-emitted event_ids, latest ingest_time wins).

Normalise all timestamps to UTC, and 
quarantine rows that can't be repaired (malformed timestamps, negative cost) instead 
of silently dropping them.

In [0]:
bronze_events_day1 = spark.table("workspace.default.bronze_events_day1")
bronze_events_day2 = spark.table("workspace.default.bronze_events_day2")
bronze_advertisers = spark.table("workspace.default.bronze_advertisers")

In [0]:
from pyspark.sql.functions import lit

events_day1_unified = (
    bronze_events_day1
    .withColumn("viewability", lit(None).cast("double"))
    .withColumn("schema_version", lit("v1"))
)

events_day2_unified = (
    bronze_events_day2
    .withColumnRenamed("media_cost", "spend")
    .withColumn("schema_version", lit("v2"))
)

events_unified = events_day1_unified.unionByName(events_day2_unified)

In [0]:
display(
    events_unified
    .select("schema_version", "event_time")
    .distinct()
    .limit(50)
)

schema_version,event_time
v1,2026-06-01T03:11:56Z
v1,2026-06-01T06:32:35Z
v1,2026-06-01T18:56:56Z
v1,2026-06-01T20:16:21Z
v1,2026-06-01T07:43:12Z
v1,2026-06-01T18:51:56Z
v1,2026-06-01T14:26:37Z
v1,2026-06-01T14:35:12Z
v1,2026-06-01T21:29:58Z
v1,2026-06-01T23:42:59Z


In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    to_timestamp,
    when,
    row_number
)

display(
    events_unified
    .filter(events_unified.schema_version == "v2")
    .select("event_time")
    .distinct()
    .limit(20)
)

event_time
2026-06-04 18:50:42+05:30
2026-06-04 20:03:11+05:30
2026-06-04 04:57:18+05:30
2026-06-04 10:59:30+05:30
2026-06-04 15:21:11+05:30
2026-06-04 11:36:19+05:30
2026-06-04 12:01:32+05:30
2026-06-04 17:45:20+05:30
2026-06-04 05:16:59+05:30
2026-06-04 19:29:29+05:30


In [0]:
spark.sql("SET spark.sql.session.timeZone").show(truncate=False)

+--------------------------+-------+
|key                       |value  |
+--------------------------+-------+
|spark.sql.session.timeZone|Etc/UTC|
+--------------------------+-------+



In [0]:
from pyspark.sql.functions import col, coalesce, try_to_timestamp, lit

events_with_ts = (
    events_unified.withColumn(
        "event_time_utc",
        coalesce(
            try_to_timestamp(col("event_time"), lit("yyyy-MM-dd'T'HH:mm:ss'Z'")),
            try_to_timestamp(col("event_time"), lit("yyyy-MM-dd HH:mm:ssXXX"))
        )
    )
)

display(
    events_with_ts
    .filter(col("event_time_utc").isNull())
    .select("schema_version", "event_id", "event_time")
)

schema_version,event_id,event_time
v1,evt_eac9aabb,N/A
v1,evt_efc9b29a,N/A
v1,evt_2efc06ff,N/A
v1,evt_25844331,N/A
v1,evt_2696dbb3,N/A
v1,evt_e04f6eb,N/A
v1,evt_cc709db3,N/A
v1,evt_d372e74f,N/A
v1,evt_7195ef4d,N/A
v1,evt_d32a6cad,N/A


In [0]:
from pyspark.sql.functions import col, when

# Rows are quarantined (not dropped) if:
#   - event_time could not be parsed into a valid UTC timestamp
#   - spend is negative
#
# This preserves bad records for audit and vendor investigation while allowing
# the rest of the pipeline to continue.

quarantine_condition = (
    col("event_time_utc").isNull() |
    (col("spend") < 0)
)

silver_events_quarantine = (
    events_with_ts
    .filter(quarantine_condition)
    .withColumn(
        "quarantine_reason",
        when(
            col("event_time_utc").isNull(),
            "malformed_event_time"
        ).when(
            col("spend") < 0,
            "negative_spend"
        )
    )
)

events_clean = (
    events_with_ts
    .filter(~quarantine_condition)
)

In [0]:
print("Malformed timestamps:",
      events_with_ts.filter(col("event_time_utc").isNull()).count())
print("Negative spend:",
      events_with_ts.filter(col("spend") < 0).count())
print("Null spend:",
      events_with_ts.filter(col("spend").isNull()).count())

Malformed timestamps: 175
Negative spend: 286
Null spend: 0


In [0]:
silver_events_quarantine.count()

461

In [0]:
events_with_ts.filter(quarantine_condition).count()

461

In [0]:
events_with_ts.filter(
    col("event_time_utc").isNull() &
    (col("spend") < 0)
).count()

0

In [0]:
#1 & 2. Dedup (exact duplicates, then latest-ingest_time-wins per event_id):
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc, col

# Step 1: drop exact duplicate rows (identical across every column)
events_deduped = events_clean.dropDuplicates()

# Step 2: for re-emitted event_ids (same id, different content/ingest_time — a
# correction, not a duplicate), keep only the latest ingest_time per event_id.
# A naive dropDuplicates() would keep an arbitrary row, potentially the stale one.
window_spec = (
    Window
    .partitionBy("event_id")
    .orderBy(
        col("ingest_time").desc(),
        col("pipeline_ingest_time").desc()
    )
)

silver_events = (
    events_deduped
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f"events_clean:                {events_clean.count()}")
print(f"after exact-dup drop:        {events_deduped.count()}")
print(f"silver_events (final):       {silver_events.count()}")

events_clean:                57155
after exact-dup drop:        55560
silver_events (final):       54460


### Silver Layer Summary

| Processing Step | Rows |
|-----------------|-----:|
|Valid records after quarantine|57,155|
|After removing exact duplicates|55,560|
|Final Silver table|54,460|

**Observations**

- 461 irreparable records were quarantined (175 malformed timestamps, 286 negative spend).

- 1,595 exact duplicate rows were removed.

- 1,100 older event versions were discarded, keeping only the latest `ingest_time` per `event_id`.

In [0]:
from pyspark.sql.functions import count

dup_check = (
    silver_events.groupBy("event_id")
    .agg(count("*").alias("cnt"))
    .filter(col("cnt") > 1)
)

print(f"Duplicate event_ids remaining in silver_events: {dup_check.count()}")

Duplicate event_ids remaining in silver_events: 0


In [0]:
silver_events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_events")

silver_events_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_events_quarantine")

In [0]:
silver_count = spark.table("workspace.default.silver_events").count()
quarantine_count = spark.table("workspace.default.silver_events_quarantine").count()

print(f"Silver Events: {silver_count}")
print(f"Silver Quarantine: {quarantine_count}")

display(spark.table("workspace.default.silver_events").limit(5))
display(spark.table("workspace.default.silver_events_quarantine").limit(5))

Silver Events: 54460
Silver Quarantine: 461


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,spend,currency,pipeline_ingest_time,source_file,viewability,schema_version,event_time_utc
evt_100426d6,impression,2026-06-06 11:38:30+05:30,2026-06-06T07:09:09.000Z,adv_1109,cmp_1109_3,banner_728x90,ctv,IN,0.0193,INR,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,0.91,v2,2026-06-06T06:08:30.000Z
evt_1004fa11,impression,2026-06-01T14:25:52Z,2026-06-01T15:45:11.000Z,adv_1130,cmp_1130_3,banner_728x90,tablet,unknown,0.006,USD,2026-07-20T15:40:45.242Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,2026-06-01T14:25:52.000Z
evt_1006546,impression,2026-06-01T06:19:43Z,2026-06-01T21:55:26.000Z,adv_1165,cmp_1165_2,banner_728x90,desktop,IN,0.0054,USD,2026-07-20T15:40:45.242Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,2026-06-01T06:19:43.000Z
evt_1006630d,impression,2026-06-05 03:33:41+05:30,2026-06-05T11:45:02.000Z,adv_1179,cmp_1179_2,banner_728x90,desktop,DE,0.0027,INR,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,0.47,v2,2026-06-04T22:03:41.000Z
evt_1007f274,impression,2026-06-03T12:46:37Z,2026-06-03T03:06:49.000Z,adv_1158,cmp_1158_3,interstitial,tablet,DE,0.0054,USD,2026-07-20T15:40:45.242Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,2026-06-03T12:46:37.000Z


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,spend,currency,pipeline_ingest_time,source_file,viewability,schema_version,event_time_utc,quarantine_reason
evt_d1541839,impression,2026-06-04 11:38:50+05:30,2026-06-04T20:23:19.000Z,adv_1123,cmp_1123_1,interstitial,desktop,CA,-0.006,USD,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,0.74,v2,2026-06-04T06:08:50.000Z,negative_spend
evt_dd2a5a74,click,2026-06-04 07:49:15+05:30,2026-06-04T01:51:53.000Z,adv_1109,cmp_1109_2,native_feed,ctv,GB,-0.2058,USD,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,0.73,v2,2026-06-04T02:19:15.000Z,negative_spend
evt_4e31c81c,impression,N/A,2026-06-04T17:01:23.000Z,adv_1116,cmp_1116_3,banner_728x90,desktop,SG,0.014,USD,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,0.57,v2,null,malformed_event_time
evt_a223a571,impression,2026-06-04 00:16:19+05:30,2026-06-04T09:00:38.000Z,adv_1130,cmp_1130_2,native_feed,ctv,IN,-0.0154,USD,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,0.69,v2,2026-06-03T18:46:19.000Z,negative_spend
evt_68a44d28,click,2026-06-04 11:17:51+05:30,2026-06-04T02:17:53.000Z,adv_1165,cmp_1165_2,banner_728x90,ctv,US,-0.8294,USD,2026-07-20T15:40:52.890Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,1.0,v2,2026-06-04T05:47:51.000Z,negative_spend


## Gold Layer – Daily Spend & Event Counts per Advertiser

Aggregate the cleaned Silver events to a daily advertiser-level summary by joining with
the advertiser dimension. 

Calculate daily event counts and total spend, converting INR
amounts to USD using a fixed exchange rate.

A left join is used to retain events whose advertiser_id is missing from the dimension.
These records are flagged as "Unknown Advertiser" instead of being dropped, ensuring
dashboard metrics remain complete while highlighting data quality issues in the
reference data.


In [0]:
silver_events = spark.table("workspace.default.silver_events")
advertisers = spark.table("workspace.default.bronze_advertisers")

In [0]:
from pyspark.sql.functions import col, when, lit

INR_TO_USD_RATE = 83.0

events_usd = silver_events.withColumn(
    "spend_usd",
    when(col("currency") == "INR", col("spend") / lit(INR_TO_USD_RATE))
    .otherwise(col("spend"))
)

In [0]:
from pyspark.sql.functions import round as _round

events_usd = events_usd.withColumn("spend_usd", _round(col("spend_usd"), 6))

In [0]:
from pyspark.sql.functions import col, when, lit

# LEFT JOIN (not INNER JOIN): retain events even if the advertiser is missing
# from the dimension table. This prevents under-reporting spend and event counts.

events_joined = (
    events_usd
    .join(advertisers, on="advertiser_id", how="left")
    .withColumn(
        "advertiser_missing_from_dim",
        col("advertiser_name").isNull()
    )
    .withColumn(
        "advertiser_name",
        when(
            col("advertiser_name").isNull(),
            lit("Unknown Advertiser")
        ).otherwise(col("advertiser_name"))
    )
)

In [0]:
print(
    "Events with missing advertiser dimension:",
    events_joined.filter(col("advertiser_missing_from_dim")).count()
)

Events with missing advertiser dimension: 9700


In [0]:
events_joined.filter(col("advertiser_missing_from_dim")) \
    .select("advertiser_id").distinct().count()

4

In [0]:
display(
    events_joined
    .filter(col("advertiser_missing_from_dim"))
    .select("advertiser_id")
    .distinct()
)

advertiser_id
adv_957
adv_1130
adv_981
adv_1165


In [0]:
display(advertisers.select("advertiser_id").distinct())

advertiser_id
adv_1109
adv_1116
adv_1123
adv_1137
adv_1144
adv_1151
adv_1158
adv_1172
adv_1179
adv_1186


In [0]:
from pyspark.sql.functions import to_date, sum as _sum, count as _count

gold_daily_advertiser_spend = (
    events_joined
    .withColumn("event_date", to_date(col("event_time_utc")))
    .groupBy("event_date", "advertiser_id", "advertiser_name", "advertiser_missing_from_dim")
    .agg(
        _sum("spend_usd").alias("total_spend_usd"),
        _count("*").alias("event_count")
    )
    .orderBy("event_date", "advertiser_id")
)

display(gold_daily_advertiser_spend.limit(20))

event_date,advertiser_id,advertiser_name,advertiser_missing_from_dim,total_spend_usd,event_count
2026-06-01,adv_1109,LoopDirect,false,35.87265399999998,670
2026-06-01,adv_1116,LoopDirect,false,36.38094399999997,718
2026-06-01,adv_1123,AltaTrips,false,44.63148299999999,662
2026-06-01,adv_1130,Unknown Advertiser,true,37.43385100000005,677
2026-06-01,adv_1137,RidgeDirect,false,30.412821000000022,693
2026-06-01,adv_1144,RidgePay,false,31.424273999999972,640
2026-06-01,adv_1151,BlueLabs,false,33.943700999999976,655
2026-06-01,adv_1158,AltaLabs,false,32.297567999999984,666
2026-06-01,adv_1165,Unknown Advertiser,true,39.05871900000002,703
2026-06-01,adv_1172,LoopWorks,false,35.80559599999995,639


In [0]:
print("Sum of event_count in gold:", gold_daily_advertiser_spend.agg(_sum("event_count")).collect()[0][0])
print("silver_events total:", silver_events.count())

Sum of event_count in gold: 54460
silver_events total: 54460


In [0]:
from pyspark.sql.functions import count

grain_check = (
    gold_daily_advertiser_spend
    .groupBy("event_date", "advertiser_id")
    .agg(count("*").alias("cnt"))
    .filter(col("cnt") > 1)
)
print("Duplicate (date, advertiser) rows:", grain_check.count())

Duplicate (date, advertiser) rows: 0


In [0]:
display(gold_daily_advertiser_spend.select("event_date").distinct().orderBy("event_date"))

event_date
2026-06-01
2026-06-02
2026-06-03
2026-06-04
2026-06-05
2026-06-06
2026-06-07


In [0]:
display(gold_daily_advertiser_spend.filter(col("total_spend_usd") <= 0))

event_date,advertiser_id,advertiser_name,advertiser_missing_from_dim,total_spend_usd,event_count


In [0]:
gold_daily_advertiser_spend.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_advertiser_spend")

print(f"gold_daily_advertiser_spend: {spark.table('workspace.default.gold_daily_advertiser_spend').count()} rows")

gold_daily_advertiser_spend: 98 rows


## Ops Notes

### Table Partitioning Strategy

**Bronze**
- Keep unpartitioned, or partition by `source_file`.
- Bronze is append-only and rarely queried directly.
- Partitioning mainly helps with selective reloads or file-level maintenance.

**Silver (`silver_events`)**
- Partition by `event_date` (derived from `event_time_utc`).
- Most queries are date-based (last 7 days, daily reprocessing, incremental loads).
- Avoid partitioning by `advertiser_id` — with hundreds/thousands of advertisers, this creates too many small partitions (small-files problem).

**Gold (`gold_daily_advertiser_spend`)**
- Partition by `event_date`.
- Table grain is already daily, and dashboards mostly filter by date ranges.

**Quarantine (`silver_events_quarantine`)**
- Partition by `event_date` for easier monitoring and troubleshooting of failed records.

### OPTIMIZE & Z-ORDER Strategy

- Incremental daily loads create many small files over time.
- Run `OPTIMIZE` periodically (nightly, or after major loads) to compact files and improve read performance.
- Z-ORDER `silver_events` on `advertiser_id` and `event_id` — these improve data skipping for:
  - Advertiser-level queries.
  - Event-level lookups during MERGE operations.
- No Z-ORDER needed for Gold:
  - Data volume is smaller.
  - Data is already pre-aggregated.

```sql
OPTIMIZE workspace.default.silver_events ZORDER BY (advertiser_id, event_id);
```

### Handling Day-8 Loads & Late-Arriving Events

**Bronze**
- Append the new file as-is.
- Maintain audit columns: `source_file`, `pipeline_ingest_time`.
- No updates or deletes in Bronze.

**Silver**
- Process the new batch through the same pipeline: parsing → validation/quarantine → deduplication.
- For late-arriving events:
  - Use `MERGE` on `event_id`.
  - Keep the row with the latest `ingest_time`.
  - Only the affected `event_date` partition is touched, not the full table.
  - Example: a correction for a June 3 event arriving on June 8 only updates the June 3 partition.

**Gold**
- Recalculate only the impacted `event_date` partitions:
  - The new partition (June 8).
  - The corrected partition (June 3).
- Avoid full-table re-aggregation.

### Data Quality Monitoring

- Advertisers missing from the dimension table are handled via `LEFT JOIN` + the `advertiser_missing_from_dim` flag, rather than being dropped.
- Monitor the missing-advertiser count over time:
  - Small, steady increase → expected (new advertisers not yet in the dimension feed).
  - Sudden spike → likely a dimension feed failure, worth investigating.

Daily spend per advertiser with a 7-day rolling average, and each day's % change vs the prior day.

In [0]:
%sql
WITH daily_with_lag AS (
  SELECT
    event_date,
    advertiser_id,
    advertiser_name,
    total_spend_usd,
    -- ROWS BETWEEN 6 PRECEDING AND CURRENT ROW = trailing 7-day window (today + 6 prior)
    AVG(total_spend_usd) OVER (
      PARTITION BY advertiser_id
      ORDER BY event_date
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_spend,
    LAG(total_spend_usd) OVER (
      PARTITION BY advertiser_id
      ORDER BY event_date
    ) AS prior_day_spend
  FROM workspace.default.gold_daily_advertiser_spend
)
SELECT
  event_date,
  advertiser_id,
  advertiser_name,
  total_spend_usd,
  ROUND(rolling_7day_avg_spend, 4) AS rolling_7day_avg_spend,
  -- NULLIF avoids a divide-by-zero error if prior_day_spend is 0; first day per
  -- advertiser has no prior day, so pct_change is NULL there by design, not a bug.
  ROUND(
    (total_spend_usd - prior_day_spend) / NULLIF(prior_day_spend, 0) * 100,
    2
  ) AS pct_change_vs_prior_day
FROM daily_with_lag
ORDER BY advertiser_id, event_date;

event_date,advertiser_id,advertiser_name,total_spend_usd,rolling_7day_avg_spend,pct_change_vs_prior_day
2026-06-01,adv_1109,LoopDirect,35.87265399999998,35.8727,null
2026-06-02,adv_1109,LoopDirect,35.23605399999999,35.5544,-1.77
2026-06-03,adv_1109,LoopDirect,36.893201999999995,36.0006,4.7
2026-06-04,adv_1109,LoopDirect,39.934838,36.9842,8.24
2026-06-05,adv_1109,LoopDirect,27.36687799999998,35.0607,-31.47
2026-06-06,adv_1109,LoopDirect,28.725070999999993,34.0048,4.96
2026-06-07,adv_1109,LoopDirect,20.99601000000001,32.1464,-26.91
2026-06-01,adv_1116,LoopDirect,36.38094399999997,36.3809,null
2026-06-02,adv_1116,LoopDirect,40.613868000000004,38.4974,11.64
2026-06-03,adv_1116,LoopDirect,42.313597,39.7695,4.19




- The first day for each advertiser (e.g., June 1) correctly returns `NULL` for
  `pct_change_vs_prior_day` because no previous day's spend exists. This is the
  expected behavior.

- `rolling_7day_avg_spend` behaves as an expanding average during the first few
  days of the time series and gradually becomes a true trailing 7-day average as
  additional daily records become available.

- Advertisers missing from the dimension table appear as **"Unknown Advertiser"**,
  confirming that the LEFT JOIN preserved event records while clearly identifying
  missing reference data.

- The query returns **98 rows (14 advertisers × 7 days)**, matching the Gold table
  exactly. This confirms that the window functions neither duplicated nor removed
  any records.

In [0]:
%sql
WITH active_days AS (

    SELECT DISTINCT
        advertiser_id,
        advertiser_name,
        event_date
    FROM workspace.default.gold_daily_advertiser_spend

),

numbered AS (

    SELECT
        advertiser_id,
        advertiser_name,
        event_date,
        ROW_NUMBER() OVER (
            PARTITION BY advertiser_id
            ORDER BY event_date
        ) AS rn
    FROM active_days

),

islands AS (

    SELECT
        advertiser_id,
        advertiser_name,
        event_date,

        DATE_SUB(event_date, rn) AS island_id

    FROM numbered

),

streaks AS (

    SELECT
        advertiser_id,
        advertiser_name,
        MIN(event_date) AS streak_start,
        MAX(event_date) AS streak_end,
        COUNT(*) AS streak_length

    FROM islands

    GROUP BY
        advertiser_id,
        advertiser_name,
        island_id

)

SELECT *
FROM streaks
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY advertiser_id
    ORDER BY streak_length DESC
) = 1
ORDER BY advertiser_id;

advertiser_id,advertiser_name,streak_start,streak_end,streak_length
adv_1109,LoopDirect,2026-06-01,2026-06-07,7
adv_1116,LoopDirect,2026-06-01,2026-06-07,7
adv_1123,AltaTrips,2026-06-01,2026-06-07,7
adv_1130,Unknown Advertiser,2026-06-01,2026-06-07,7
adv_1137,RidgeDirect,2026-06-01,2026-06-07,7
adv_1144,RidgePay,2026-06-01,2026-06-07,7
adv_1151,BlueLabs,2026-06-01,2026-06-07,7
adv_1158,AltaLabs,2026-06-01,2026-06-07,7
adv_1165,Unknown Advertiser,2026-06-01,2026-06-07,7
adv_1172,LoopWorks,2026-06-01,2026-06-07,7


- Every advertiser has activity on each of the seven days in the assessment data.
- Therefore, the longest consecutive active streak for every advertiser is 7 days (2026-06-01 to 2026-06-07).
- The query correctly identifies continuous activity using the standard Gaps-and-Islands pattern, even though this sample dataset contains no gaps.
- Advertisers missing from the dimension table are retained as "Unknown Advertiser", confirming that the LEFT JOIN behavior is preserved.

### MERGE Statement for Late-Arriving Corrections

The following Delta Lake `MERGE` statement demonstrates how a late-arriving
corrections batch would be upserted into the Silver table.

**Assumptions**
- The incoming corrections batch has already been loaded into a staging table named
  `workspace.default.late_arriving_corrections`.
- Existing records are updated only when the incoming row has a more recent `ingest_time`.
- New `event_id`s are inserted into the Silver table.

```sql
MERGE INTO workspace.default.silver_events AS target

USING workspace.default.late_arriving_corrections AS source

ON target.event_id = source.event_id

WHEN MATCHED
AND source.ingest_time > target.ingest_time
THEN UPDATE SET
    target.event_type = source.event_type,
    target.event_time = source.event_time,
    target.event_time_utc = source.event_time_utc,
    target.ingest_time = source.ingest_time,
    target.advertiser_id = source.advertiser_id,
    target.campaign_id = source.campaign_id,
    target.placement = source.placement,
    target.device = source.device,
    target.geo = source.geo,
    target.spend = source.spend,
    target.currency = source.currency,
    target.pipeline_ingest_time = source.pipeline_ingest_time,
    target.source_file = source.source_file,
    target.viewability = source.viewability,
    target.schema_version = source.schema_version

WHEN NOT MATCHED
THEN INSERT (
    event_id,
    event_type,
    event_time,
    ingest_time,
    advertiser_id,
    campaign_id,
    placement,
    device,
    geo,
    spend,
    currency,
    pipeline_ingest_time,
    source_file,
    viewability,
    schema_version,
    event_time_utc
)
VALUES (
    source.event_id,
    source.event_type,
    source.event_time,
    source.ingest_time,
    source.advertiser_id,
    source.campaign_id,
    source.placement,
    source.device,
    source.geo,
    source.spend,
    source.currency,
    source.pipeline_ingest_time,
    source.source_file,
    source.viewability,
    source.schema_version,
    source.event_time_utc
);
```

## Reusable Data Quality Utility

This utility performs configurable data quality checks against a PySpark DataFrame.
Rules are passed declaratively, making the framework reusable across datasets without
changing the implementation.

Supported checks:
- Null-rate threshold
- Column uniqueness
- Freshness (maximum event age)

In [0]:
def run_dq_checks(df, rules):
    """
    Run configurable data quality checks against a PySpark DataFrame.

    Parameters
    ----------
    df : pyspark.sql.DataFrame
        Input dataframe.

    rules : list[dict]
        Declarative quality rules.

    Returns
    -------
    list[dict]
        Structured pass/fail report.
    """

    report = []

    total_rows = df.count()

    for rule in rules:

        rule_type = rule["type"]

        # -----------------------------
        # Null-rate check
        # -----------------------------
        if rule_type == "null_rate":

            column = rule["column"]
            threshold = rule["threshold"]

            null_count = (
                df.filter(col(column).isNull()).count()
            )

            null_rate = null_count / total_rows

            report.append({
                "check": "null_rate",
                "column": column,
                "value": round(null_rate,4),
                "threshold": threshold,
                "status": "PASS" if null_rate <= threshold else "FAIL"
            })

        # -----------------------------
        # Uniqueness
        # -----------------------------
        elif rule_type == "unique":

            column = rule["column"]

            duplicates = (
                total_rows -
                df.select(column).distinct().count()
            )

            report.append({

                "check":"unique",

                "column":column,

                "duplicate_rows":duplicates,

                "status":"PASS" if duplicates==0 else "FAIL"

            })

        # -----------------------------
        # Freshness
        # -----------------------------
        elif rule_type == "freshness":

            column = rule["column"]

            max_age_days = rule["max_age_days"]

            latest = df.select(
                spark_max(column)
            ).collect()[0][0]

            age_days = (
                datetime.now(timezone.utc) -
                latest.replace(tzinfo=timezone.utc)
            ).days

            report.append({

                "check":"freshness",

                "column":column,

                "age_days":age_days,

                "threshold":max_age_days,

                "status":"PASS" if age_days <= max_age_days else "FAIL"

            })

    return report

In [0]:
dq_rules = [

    {
        "type":"null_rate",
        "column":"spend",
        "threshold":0.01
    },

    {
        "type":"unique",
        "column":"event_id"
    },

    {
        "type":"freshness",
        "column":"event_time_utc",
        "max_age_days":365
    }

]

In [0]:
from pyspark.sql.functions import col, count, when, max as spark_max
from datetime import datetime, timezone

dq_report = run_dq_checks(
    silver_events,
    dq_rules
)

for result in dq_report:
    print(result)

{'check': 'null_rate', 'column': 'spend', 'value': 0.0, 'threshold': 0.01, 'status': 'PASS'}
{'check': 'unique', 'column': 'event_id', 'duplicate_rows': 0, 'status': 'PASS'}
{'check': 'freshness', 'column': 'event_time_utc', 'age_days': 42, 'threshold': 365, 'status': 'PASS'}


In [0]:
#Test 1 – Null threshold passes

result = run_dq_checks(
    silver_events,
    [{
        "type":"null_rate",
        "column":"spend",
        "threshold":0.01
    }]
)

assert result[0]["status"] == "PASS"

In [0]:
#Test 2 – event_id uniqueness
result = run_dq_checks(
    silver_events,
    [{
        "type":"unique",
        "column":"event_id"
    }]
)

assert result[0]["status"] == "PASS"

In [0]:
#Test 3 – Freshness edge case
result = run_dq_checks(
    silver_events,
    [{
        "type":"freshness",
        "column":"event_time_utc",
        "max_age_days":1
    }]
)

assert result[0]["status"] == "FAIL"

In [0]:
dq_report = run_dq_checks(silver_events, dq_rules)

for result in dq_report:
    print(result)

{'check': 'null_rate', 'column': 'spend', 'value': 0.0, 'threshold': 0.01, 'status': 'PASS'}
{'check': 'unique', 'column': 'event_id', 'duplicate_rows': 0, 'status': 'PASS'}
{'check': 'freshness', 'column': 'event_time_utc', 'age_days': 42, 'threshold': 365, 'status': 'PASS'}


DSP Medallion Pipeline 
I built this using the Medallion architecture — Bronze, Silver, Gold — on Databricks with PySpark and Delta Lake. Bronze ingests the raw vendor CSV files as-is, without touching the business data. Since the two event files come in with different schemas, I kept them as separate Delta tables rather than forcing a merge at this layer, and only added lineage metadata like pipeline_ingest_time and source_file. 

Silver is where the schema drift actually gets reconciled. I renamed media_cost to spend, added viewability as NULL for v1 records where it doesn't exist, and unified both event streams into one table. Timestamps are normalized to UTC, and anything malformed or carrying negative spend gets quarantined rather than dropped or guessed at. For duplicates, exact copies are removed outright, and where the same event_id shows up more than once, I keep whichever record has the latest ingest_time. 

Gold joins the cleaned events against the advertiser dimension to produce daily advertiser-level spend and event counts, converting INR to USD along the way. If an event's advertiser isn't in the dimension table, I don't drop it — I flag it as an Unknown Advertiser and keep it in the output so nothing silently disappears from reporting. A few assumptions worth calling out: I used a fixed exchange rate of 1 USD = 83 INR, ingest_time decides which version of a record wins, and both negative spend and malformed timestamps are treated as unrecoverable and quarantined rather than auto-corrected. 

The main trade-off is that Bronze intentionally doesn't enforce a unified schema — that's deferred to Silver, which keeps ingestion resilient to vendor-side changes. The fixed FX rate is a simplification I'd swap for a daily FX lookup table in production. And quarantining bad records instead of silently fixing or dropping them was a deliberate choice — it keeps the pipeline auditable and keeps garbage data out of downstream reporting, even at the cost of some records not making it through on the first pass.

Subject: Vendor File Issue – Dashboard Delay

Hi Team,

Wanted to flag something from tonight's data run — we found corruption in today's vendor file during ingestion, and it's not safe to process as-is. Because of that, tomorrow's 9:00 AM dashboard refresh is going to be delayed.

We're already in touch with the vendor to get a corrected file over, and as soon as it lands, we'll rerun the pipeline right away. We'll keep you posted as things move and share a revised time estimate as soon as we have one.

Sorry for the disruption — we know timing matters here.

Thanks,
Data Engineering Team

Code Review

def load_events(spark, path):
    df = spark.read.csv(path)
    df = df.dropDuplicates()
    df = df.filter(df.spend != None)
    for row in df.collect():
        if row['spend'] < 0:
            df = df.filter(df.event_id != row['event_id'])
    return df.cache()


A few things need fixing before this moves forward:

Schema isn't defined upfront — CSV is read without headers or explicit schema, causing wrong data types and slow inference.

dropDuplicates() only catches exact duplicate rows, not multiple versions of the same event_id. Needs explicit logic to pick the right version.

Null filter bug: df.spend != None isn't valid PySpark. Should be col("spend").isNotNull().

Bigger issue — collect() pulls all rows to the driver. Won't scale, risks driver memory failures on large data.

Row-by-row iteration defeats Spark's distributed model entirely — slow and inefficient.

DataFrame is also rebuilt inside the loop, re-running the same filtering unnecessarily.

Caching is applied without clear reuse — wastes cluster memory when the DataFrame isn't reused multiple times.

Bottom line: rewrite using Spark transformations, not driver-side loops.

Scaling the Pipeline

If the feed grew to 10x its current size and started arriving hourly, here's what I'd prioritize:

Switch to Auto Loader or incremental ingestion instead of scanning full files each run. Partition Delta tables by event_date to cut down scan costs. 

Apply OPTIMIZE and ZORDER on columns we filter on most — mainly event_date and advertiser_id. Move to incremental MERGE operations so we're only processing new and late-arriving data, not reprocessing everything.
 
Tune Spark partitioning and cluster sizing to match the larger workload. Replace the fixed FX rate with a proper reference exchange-rate table. And add monitoring, alerting, and automated retries so failed runs get caught and handled without manual intervention.